# Export images as TIFs

gg-napari-env

In [1]:
from napari_czifile2 import napari_get_reader
import numpy as np
import tifffile as tiff
import numpy as np
from tqdm import tqdm
import sys 
import os 

In [2]:
raw_data_dirs = os.listdir('../../../RNA-FISH-raw-data/')
raw_data_p14 = [d for d in raw_data_dirs if '20250813' in d]
raw_data_p14

['20250813 C P14 U34-B3-488 Chymotrypsin-B2-647 DAPI',
 '20250813 B P14 U34-B3-488 Lnc4-B1-546 U21-B5-647 DAPI',
 '20250813 D P14 9E118-exons-B1-546 9E118-introns-B2-647 9E129-B3-488 DAPI',
 '20250813 A P14 P1-B1-546 P2-B2-647 P1-antisense-B3-488 DAPI']

In [3]:
for input in raw_data_p14: 
    # Check if input directory exists
    input_dir = f'../../../RNA-FISH-raw-data/{input}/'
    assert os.path.exists(input_dir), 'Input directory does not exist'
    czi_files = [f for f in os.listdir(input_dir) if f.endswith('.czi')]
    print(f"Found {len(czi_files)} czi files in {input_dir}")
    print(czi_files)

    # Create output directory 
    output_dir = f'../../../RNA-FISH-raw-data/tifs/{input}/'
    os.makedirs(output_dir, exist_ok=True)

    for f in czi_files:
        print(f"processing {f}")
        
        file_path = os.path.join(input_dir, f)
        reader = napari_get_reader(file_path)
        if reader is not None:
            layer_data = reader(file_path)
            image_data, metadata, layer_type = layer_data[0]

            # Remove singleton dimension 
            image_data = np.squeeze(image_data)  
            print("Metadata:", metadata)
            print("Image shape:", image_data.shape)  

            # Export image as TIF 
            tif_path = os.path.join(output_dir, f.replace('.czi', '.tif'))
            print(f"Exporting {f} to {tif_path}")
            tiff.imwrite(tif_path, 
                            image_data, # Image data from CZI file
                            metadata=metadata, # Metadata from CZI file
                            dtype=image_data.dtype, # 8-bit 
                            compression='deflate', # lossless zlib
                            tile=(256, 256) # Tile size for TIF export, adjust as needed
                        ) 
                
            

Found 8 czi files in ../../../RNA-FISH-raw-data/20250813 C P14 U34-B3-488 Chymotrypsin-B2-647 DAPI/
['20250813 C sample 8 stack.czi', '20250813 C sample 5 stack.czi', '20250813 C sample 4 stack.czi', '20250813 C sample 7 stack.czi', '20250813 C sample 6 stack.czi', '20250813 C sample 1 stack.czi', '20250813 C sample 3 stack.czi', '20250813 C sample 2 stack.czi']
processing 20250813 C sample 8 stack.czi
Metadata: {'rgb': False, 'channel_axis': 2, 'translate': (0.0, 0.0, 0.0, 0.0), 'scale': (1.0, 1.0, 0.0974884033203125, 0.0974884033203125), 'contrast_limits': None, 'name': ['AF488-T1', 'AF647-T1', 'DAPI-T2']}
Image shape: (92, 3, 2048, 2048)
Exporting 20250813 C sample 8 stack.czi to ../../../RNA-FISH-raw-data/tifs/20250813 C P14 U34-B3-488 Chymotrypsin-B2-647 DAPI/20250813 C sample 8 stack.tif
processing 20250813 C sample 5 stack.czi
Metadata: {'rgb': False, 'channel_axis': 2, 'translate': (0.0, 0.0, 0.0, 0.0), 'scale': (1.0, 1.0, 0.0974884033203125, 0.0974884033203125), 'contrast_limi